# Predictive Analysis- Data Preprocessing
This notebook preprocesses our dataset and merges the 5 individual datasets into a master dataset, with target variables for 24-hour failure prediction (classification).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

### Loading all datasets

In [6]:
DATA_PATH = "drive/Othercomputers/My Mac/predictive-analysis/data/raw/"

telemetry_df = pd.read_csv(f'{DATA_PATH}PdM_telemetry.csv', parse_dates=['datetime'])
errors_df = pd.read_csv(f'{DATA_PATH}PdM_errors.csv', parse_dates=['datetime'])
maintenance_df = pd.read_csv(f'{DATA_PATH}PdM_maint.csv', parse_dates=['datetime'])
failures_df = pd.read_csv(f'{DATA_PATH}PdM_failures.csv', parse_dates=['datetime'])
machines_df = pd.read_csv(f'{DATA_PATH}PdM_machines.csv')

### Data filtering
Our dataset has maintainance data from 2014 but other data (like failure, telementry) starts from 2015. So, we need to filter our maintainace data to match other dataset and start from 2015.

In [7]:
print(f"Before: {len(maintenance_df):,} rows")
print(f"Date range: {maintenance_df['datetime'].min()} to {maintenance_df['datetime'].max()}")

maintenance_df = maintenance_df[maintenance_df['datetime'] >= '2015-01-01']

print(f"After: {len(maintenance_df):,} rows")
print(f"Date range: {maintenance_df['datetime'].min()} to {maintenance_df['datetime'].max()}")

Before: 3,286 rows
Date range: 2014-06-01 06:00:00 to 2016-01-01 06:00:00
After: 2,886 rows
Date range: 2015-01-01 06:00:00 to 2016-01-01 06:00:00


### Preparing events data

In this part, we are processing events data (errors, maintenance, failures) for each machine at every hour, and creating 3 pivot tables where each row represents the event status of a specific machine at a specific time. These tables include binary flags for each event type showing whether any event occurred during that time.
<br>This structure makes it easier to analyze a machine's condition over time: "when & what event happened to this machine?" and will be merge into master dataset later.


In [9]:
# Processing ERROR events
# binary flags for each error type
errors_pivot = errors_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='errorID',
    aggfunc='size',
    fill_value=0
).reset_index()

# renaming columns
errors_pivot.columns = ['machineID', 'datetime'] + [f'{col}' for col in errors_pivot.columns[2:]] # setting error type columns from pivoted table

# error flag
errors_pivot['has_error'] = (errors_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Error pivot columns: {list(errors_pivot.columns)}")


Error pivot columns: ['machineID', 'datetime', 'error1', 'error2', 'error3', 'error4', 'error5', 'has_error']


In [11]:
# Processing MAINTENANCE events
maintenance_pivot = maintenance_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='comp',
    aggfunc='size',
    fill_value=0
).reset_index()

# rename columns
maintenance_pivot.columns = ['machineID', 'datetime'] + [f'maint_{col}' for col in maintenance_pivot.columns[2:]]

# maintenance flag
maintenance_pivot['has_maintenance'] = (maintenance_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"Maintenance columns: {list(maintenance_pivot.columns)}")


Maintenance columns: ['machineID', 'datetime', 'maint_comp1', 'maint_comp2', 'maint_comp3', 'maint_comp4', 'has_maintenance']


In [14]:
# Processing FAILURE events
failures_pivot = failures_df.pivot_table(
    index=['machineID', 'datetime'],
    columns='failure',
    aggfunc='size',
    fill_value=0
).reset_index()

# rename columns
failures_pivot.columns = ['machineID', 'datetime'] + [f'failure_{col}' for col in failures_pivot.columns[2:]]

# failure flag
failures_pivot['has_failure'] = (failures_pivot.iloc[:, 2:].sum(axis=1) > 0).astype(int)

print(f"   Failure columns: {list(failures_pivot.columns)}")

   Failure columns: ['machineID', 'datetime', 'failure_comp1', 'failure_comp2', 'failure_comp3', 'failure_comp4', 'has_failure']


### Merging dataset